## Prerequisite Code

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../initial-setup/03-utils

In [0]:
# Create widgets
dbutils.widgets.text('catalog', 'sportsdirect_sales', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

# Access widgets
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
# Define source directory
source_dir = f's3://sd-warrior-acquisition/orders/landing/*.csv'

# Define target directory
target_dir = f's3://sd-warrior-acquisition/orders/archive'

## Warrior Bronze Layer

In [0]:
# Get the raw data
raw_data = spark.read \
    .format('csv') \
    .option('header', True) \
    .option('inferSchema', True) \
    .load(source_dir) \
    .withColumn('read_timestamp', F.current_timestamp()) \
    .select('*', '_metadata.file_name', '_metadata.file_size')

In [0]:
# Write raw data to the bronze table
raw_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', True) \
            .mode('overwrite') \
                .saveAsTable(f'{catalog}.{wr_bronze_schema}.fact_order')

In [0]:
# Move processed files to the archive folder
processed_files = dbutils.fs.ls('s3://sd-warrior-acquisition/orders/landing/')

for file in processed_files:
    if file.name.endswith('.csv'):
        dbutils.fs.mv(file.path, f'{target_dir}/{file.name}', True)

## Warrior Silver Layer

In [0]:
# Clean data and apply transformations

transformed_data = spark.sql(f'SELECT * FROM {catalog}.{wr_bronze_schema}.fact_order')

# Remove the weekday from order_placement_date
transformed_data = transformed_data \
    .withColumn(
        'order_placement_date',
        F.regexp_replace(
            'order_placement_date',
            r'([A-Za-z]+, )?',
            ''
        )
    )

# Convert order_placement_date to date
transformed_data = transformed_data \
    .withColumn(
        'order_placement_date',
        F.coalesce(
            F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
            F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
            F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
            F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
        )
    )

# Rename order_placement_date to date
transformed_data = transformed_data \
    .withColumnRenamed(
        'order_placement_date',
        'date'
    )

# Replace invalid customer_ids with '999999'
transformed_data = transformed_data \
    .withColumn(
        'customer_id',
        F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
        .otherwise('999999')
    )

# Rename customer_id to customer_code
transformed_data = transformed_data \
    .withColumnRenamed(
        'customer_id',
        'customer_code'
    )

# Convert product_id to string
transformed_data = transformed_data \
    .withColumn(
        'product_id',
        F.col('product_id').cast('string')
    )

# Add product codes
transformed_data = transformed_data \
    .join(
        spark.sql(f'SELECT * FROM {catalog}.{wr_silver_schema}.dim_product'),
        on='product_id',
        how='inner'
    ) \
        .select(
            transformed_data['order_id'],
            transformed_data['date'],
            transformed_data['customer_code'],
            transformed_data['product_id'],
            transformed_data['order_qty'],
            transformed_data['read_timestamp'],
            transformed_data['file_name'],
            transformed_data['file_size'],
            F.col('dim_product.product_code').alias('product_code')
        )

# Remove orders with no quantity
transformed_data = transformed_data.filter(F.col('order_qty') > 0)

# Convert order_qty to int
transformed_data = transformed_data \
    .withColumn(
        'order_qty',
        F.col('order_qty').cast('int')
    )

# Rename order_qty to sold_quantity
transformed_data = transformed_data \
    .withColumnRenamed(
        'order_qty',
        'sold_quantity'
    )

# Delete duplicate orders
transformed_data = transformed_data.dropDuplicates(['order_id', 'date', 'customer_code', 'product_code', 'sold_quantity'])

In [0]:
# Verify transformed data
display(transformed_data)

order_id,date,customer_code,product_id,sold_quantity,read_timestamp,file_name,file_size,product_code
FJUL33321602,2025-07-01,789321,25891103,419,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909
FJUL33321602,2025-07-01,789321,25891602,74,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,836744df97fd09ea8a22b4693e616edd9fbda3f34a1262dcf36b75148f80c6ce
FJUL33503502,2025-07-01,789503,25891202,214,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,62254dca28e1f3ce45668a4abc8571ad4fc3923df56d7a0c12c369dc76aa1f67
FJUL34220601,2025-07-01,789220,25891203,235,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,7cde4fee80f465659932d7e9336957d070865e23db86fb63dd73a40dc309f430
FJUL32203402,2025-07-01,789203,25891303,64,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5
FJUL34103403,2025-07-01,789103,25891301,67,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,d5f5123af988ee1eb64fa302fdbc78b2965b9d0f9d0e68d51c50bfcaa30d6210
FJUL32202402,2025-07-01,789202,25891303,40,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5
FJUL32422401,2025-07-01,789422,25891401,342,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2
FJUL32720302,2025-07-01,789720,25891102,402,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f
FJUL32902402,2025-07-01,789902,25891302,70,2026-03-18T06:42:57.037Z,orders_2025_07_01.csv,20744,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c


In [0]:
# Write transformed data to the silver table
transformed_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', 'true') \
            .option('mergeSchema', 'true') \
                .mode('overwrite') \
                    .saveAsTable(f'{catalog}.{wr_silver_schema}.fact_order')

## Warrior Gold Layer

In [0]:
# Get the transformed data
analytics_data = spark.sql(f'SELECT * FROM {catalog}.{wr_silver_schema}.fact_order')

In [0]:
# Select necessary columns
analytics_data = analytics_data.select('date', 'product_code', 'customer_code', 'sold_quantity')

In [0]:
# View analytics data and load to the gold table

display(analytics_data)

analytics_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', 'true') \
            .option('mergeSchema', 'true') \
                .mode('overwrite') \
                    .saveAsTable(f'{catalog}.{wr_gold_schema}.fact_order')

date,product_code,customer_code,sold_quantity
2025-07-01,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909,789321,419
2025-07-01,836744df97fd09ea8a22b4693e616edd9fbda3f34a1262dcf36b75148f80c6ce,789321,74
2025-07-01,62254dca28e1f3ce45668a4abc8571ad4fc3923df56d7a0c12c369dc76aa1f67,789503,214
2025-07-01,7cde4fee80f465659932d7e9336957d070865e23db86fb63dd73a40dc309f430,789220,235
2025-07-01,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,789203,64
2025-07-01,d5f5123af988ee1eb64fa302fdbc78b2965b9d0f9d0e68d51c50bfcaa30d6210,789103,67
2025-07-01,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,789202,40
2025-07-01,958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2,789422,342
2025-07-01,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f,789720,402
2025-07-01,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c,789902,70


## Sports Direct Gold Layer

In [0]:
# Merge data from the Warrior gold table to the Sports Direct gold table

wr_fact_order_table = spark.sql(f'SELECT * FROM {catalog}.{wr_gold_schema}.fact_order')

# Aggregate sold_quantity by month
wr_fact_order_table = wr_fact_order_table \
    .withColumn(
        'date',
        F.trunc('date', 'MM')
    ) \
        .groupBy(
            'date',
            'customer_code',
            'product_code'
        ) \
            .agg(
                F.sum('sold_quantity').alias('sold_quantity')
            )

# Select necessary columns
wr_fact_order_table = wr_fact_order_table.select('date', 'product_code', 'customer_code', 'sold_quantity')

# Get the Sports Direct gold table
sd_fact_order_table = DeltaTable.forName(spark, f'{catalog}.{sd_gold_schema}.fact_order')

# Merge data
sd_fact_order_table.alias('target').merge(
    wr_fact_order_table.alias('source'),
    'target.date = source.date AND target.product_code = source.product_code AND target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(wr_fact_order_table)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8206571139731506>, line 1
----> 1 display(wr_fact_order_table)

NameError: name 'wr_fact_order_table' is not defined